# FDSI benchmark: optimisation -- sv_loss (TPE, median aggregation)

Runs an Optuna TPE hyperparameter search (`optimize_adapt_decomp_pooled_memory`) over
`wh_learning_rate` and `sv_learning_rate`, pooling the
3 triangular conditions (`POOL_CONDITIONS`, ramp durations from 5 s to 40 s) so one search covers
all of them at once. Each trial's `AdaptDecomp` run is scored on a single objective:
separation-vector loss (summed across the pool) and the trial with the lowest
value wins, separately for each `lr_mode`.

**Load-only by default** (`RUN_OPTIMISATION=False`) -- reads the cached `study.pkl` under
`OUTPUTS_ROOT/adaptation/optimisation/<lr_mode>/tpe_sv_median/`, does not re-run the search.

## Config

In [1]:
RUN_OPTIMISATION = False   # run the pooled tpe_sv_median study, or reuse a cached study.pkl

import yaml
from pathlib import Path

with open('../../configs/data_configs/fdsi_benchmark_grid.yaml') as f:
    grid = yaml.safe_load(f)

DATA_DIR = Path(grid['data_root'])           # Raw archive root: <sub>/{clean,noisy}/
OUTPUTS_ROOT = Path(grid['outputs_root'])    # Precomputed-outputs archive root
CAL_DIR, ADAPT_DIR = OUTPUTS_ROOT / 'calibration', OUTPUTS_ROOT / 'adaptation'
OPT_DIR = ADAPT_DIR / 'optimisation'

FS, EXT_FACT = grid['fs'], grid['ext_fact']
TOL_SPIKE_MS, ROA_CAL_TH = grid['tol_spike_ms'], grid['roa_cal_th']
CAL_END, ISO_DUR = int(grid['cal_duration_s'] * FS), int(grid['iso_duration_s'] * FS)

SUBJECTS, CONDITIONS, SNR_LEVELS = grid['subjects'], grid['conditions'], grid['snr_levels']
TRIANGULAR = [c for c in CONDITIONS if 'triangular' in c]

POOL_CONDITIONS = grid['pool_conditions']
HOLDOUT_CONDITIONS = [c for c in CONDITIONS if c not in POOL_CONDITIONS]
OPTIM_SUB, OPTIM_SNR, N_TRIALS = grid['optim_sub'], grid['optim_snr'], grid['n_trials']

BASE_CONFIG_DIR = Path('../../configs/adapt_configs')
BASE_CONFIG_PATH = BASE_CONFIG_DIR / 'default_muniverse.yaml'   # wh_sv_coupling=True; lr_mode overridden per branch below
LR_TOKEN_TO_MODE = {'lr_fixed': 'fixed', 'lr_relerror': 'rel_error'}

print(f'Apply-to-all grid: {len(SUBJECTS)} subjects x {len(CONDITIONS)} conditions x {len(SNR_LEVELS)} SNR levels '
      f'= {len(SUBJECTS) * len(CONDITIONS) * len(SNR_LEVELS)} recordings')

OBJECTIVE = 'sv_loss'
SAMPLER_NAME = 'tpe_sv_median'

Apply-to-all grid: 5 subjects x 5 conditions x 4 SNR levels = 100 recordings


## Data pool

The 3 pooled conditions' `PooledDatasetMemory`s, loaded via `load_data()` from `configs/data_configs/fdsi_pool_memory_example.yaml` -- see [optimisation.md](../../docs/optimisation.md#loading-a-pool-from-a-data_config-yaml).

In [2]:
import pickle
import shutil
import sys
from typing import Dict
sys.path.insert(0, '../..')
sys.path.insert(0, str(Path.cwd()))

import optuna
from adapt_decomp import AdaptationResult, load_data
from adapt_decomp.adaptation import AdaptConfig
from adapt_decomp.adaptation.optimize import DEFAULT_PARAM_SPACE, optimize_adapt_decomp_pooled_memory
from adapt_decomp.utils.plots import plot_optimisation_landscape_grid
import fdsi_common as fc

DATA_CONFIG_PATH = Path('../../configs/data_configs/fdsi_pool_memory_example.yaml')
with DATA_CONFIG_PATH.open() as f:
    data_config = yaml.safe_load(f)
pool = load_data(data_config)
assert set(pool) == set(POOL_CONDITIONS), pool.keys()
{name: tuple(cond.emg.shape) for name, cond in pool.items()}

{'triangular-ramp40s': (184320, 100),
 'triangular-ramp10s': (184320, 100),
 'triangular-ramp5s': (184320, 100)}

## Optimisation

In [3]:
studies: Dict[str, optuna.Study] = {}
best_configs: Dict[str, AdaptConfig] = {}
best_outputs: Dict[str, dict] = {}

for lr_token, lr_mode in LR_TOKEN_TO_MODE.items():
    base_config = AdaptConfig.from_yaml(BASE_CONFIG_PATH)
    base_config.lr_mode = lr_mode   # default_muniverse.yaml is lr_mode='fixed' -- override for lr_relerror
    best_dir = OPT_DIR / lr_token / SAMPLER_NAME
    study_path = best_dir / 'study.pkl'

    n_complete = 0
    if study_path.exists():
        with open(study_path, 'rb') as f:
            cached_study = pickle.load(f)
        n_complete = sum(t.state.name == 'COMPLETE' for t in cached_study.trials)

    if n_complete >= N_TRIALS:
        print(f'{lr_token}/{SAMPLER_NAME}: loading cached study ({n_complete} complete trials).')
        study = cached_study
        best_config = AdaptConfig.from_yaml(BASE_CONFIG_PATH)
        best_config.lr_mode = lr_mode
        for k, v_ in study.best_params.items():
            setattr(best_config, k, v_)
        outputs = {c: AdaptationResult.load(best_dir / f'{c}.pkl')
                   for c in POOL_CONDITIONS if (best_dir / f'{c}.pkl').exists()}
    elif RUN_OPTIMISATION:
        shutil.rmtree(best_dir, ignore_errors=True)
        print(f'Running {lr_token}/{SAMPLER_NAME} ({N_TRIALS} trials, objective={OBJECTIVE}) ...')
        outputs, best_config, study = optimize_adapt_decomp_pooled_memory(
            pool=pool, param_space=DEFAULT_PARAM_SPACE, objective=OBJECTIVE,
            base_config=base_config, compute_roa=True, roa_kwargs={'tol_spike_ms': TOL_SPIKE_MS},
            n_trials=N_TRIALS, random_seed=42, best_result_path=str(best_dir),
        )
    else:
        raise FileNotFoundError(f'No cached {lr_token}/{SAMPLER_NAME} study and RUN_OPTIMISATION=False.')

    studies[lr_token] = study
    best_configs[lr_token], best_outputs[lr_token] = best_config, outputs
    print(f'  {lr_token}: wh_learning_rate={best_config.wh_learning_rate:.4g}  '
          f'sv_learning_rate={best_config.sv_learning_rate:.4g}  best_value={study.best_value:.4g}')

lr_fixed/tpe_sv_median: loading cached study (50 complete trials).
  lr_fixed: wh_learning_rate=0.04492  sv_learning_rate=0.001377  best_value=241.5
lr_relerror/tpe_sv_median: loading cached study (50 complete trials).
  lr_relerror: wh_learning_rate=0.0004008  sv_learning_rate=0.01795  best_value=205.6


### Optimisation landscape -- RoA vs pooled wh_loss/sv_loss/total_loss

In [4]:
best_trials = {t: studies[t].best_trial for t in studies}
fig1 = plot_optimisation_landscape_grid(studies, {t: t for t in studies}, best_trials)
fig1.show()

## Promote the winning `lr_fixed` config

Saves this branch's winning **`lr_fixed`**-mode config as a first-class, reusable
`configs/adapt_configs/` entry -- `04b_sv_loss_application.ipynb` (and
`05_comparison_sv_loss_pareto_roa.ipynb`) read it back via `AdaptConfig.from_yaml(...)` instead of
reaching into `OUTPUTS_ROOT/adaptation/optimisation/...`. Only `lr_fixed` is promoted this way;
`lr_relerror` still reads back from its own cached `config.yaml` under `OUTPUTS_ROOT`, same as
before.

In [5]:
promoted_path = Path('../../configs/adapt_configs/optim_muniverse_fdsi_sv.yaml')
best_configs['lr_fixed'].to_yaml(promoted_path)
print(f'Promoted lr_fixed config -> {promoted_path}')

Promoted lr_fixed config -> ..\..\configs\adapt_configs\optim_muniverse_fdsi_sv.yaml
